# GENAI – Modül 2 – Case Study: Sales Forecast Adjustment

## Satış Tahminlerinde Karar Destek Sistemi

Büyük perakende / e-ticaret. Günlük satıştan **30 günlük** forecast çıkıyor. Bu sayı stoka, lojistiğe, kampanya bütçesine, tedarikçi sözleşmesine gidiyor. Yani “model biraz sapmış, idare eder” demek kolay değil.

Elimizde ~**95.000** kayıt var. Ana model **LSTM**. Trend, mevsimsellik, kampanya etkisini buradan öğreniyor. SMAPE **13.2**. Kabul edilir denmiş, ama özel gün, ani kampanya, hava, rakip, bölgesel etkinlik gelince hata büyüyor.

Şirket LSTM’in sayısını silip LLM’e yeniden kestirtmek istemiyor. **Human-in-the-Loop**: LSTM üretir, LLM yorumlar, insan onaylar. Pilotta LLM ayarından sonra SMAPE **12.6** olmuş. 0.6 puan. Güzel duruyor ama sürdürülebilir mi, açıklanabilir mi, operasyona nasıl biner, henüz net değil.

Soru “LLM daha mı zeki?” değil. Tahmini kim üretecek, kim yorumlayacak, canlıya bu haliyle girilir mi.


# Görev 1 – Model ve LLM Rol Ayrımı

## Soru

Bu senaryoda LSTM modeli ile LLM’in rolleri neden net bir şekilde ayrılmalıdır?

LLM’in doğrudan tahmin üreten bir model olarak kullanılması hangi riskleri doğurur?


## Cevap

LSTM burada sayı üreten taraf. 95 bin günlük satıştan trendi, mevsimselliği, kampanya etkisini öğrenmiş. Çıktısı 30 günlük forecast. Kaybı da belli: SMAPE 13.2. Aynı eğitim, aynı girdi; model aşağı yukarı aynı sayıyı basar. Zaman serisi öğrenimi bu. Deterministik demiyorum, dropout falan vardır, ama en azından ölçülebilir bir regresyon modeli.

LLM ise token tahmin ediyor. Dil. “Yarın 14.200 birim” demesi, o sayıyı zaman serisinden çıkardığı anlamına gelmez. Cümle kuruyor. Temperature 0.7’de aynı prompt’a başka gerekçe, başka sapma yazabilir. İş kritik kararın altına bunu koymak başka.

Roller karışırsa iki şey kaybolur. Bir: hata gelince kim sapıttı bakamazsın. LSTM mi 13.2’de kaldı, LLM mi “hava kötü olur” diye %8 indirdi. İki: açıklanabilirlik. Case zaten `explainability_required` diyor. LSTM’in SMAPE’si var. LLM’in cümlesi her seferinde değişebilir. İnsan “neden değişti” diye sorduğunda dün başka, bugün başka metin görür.

LLM’i doğrudan forecast modeli yapmanın riski net:

- **Hallüsinasyon.** Context’te hava, rakip kampanyası, bölgesel etkinlik yok. Model yoksa uydurur, sayıyı kaydırır. Stok şişer ya da raf boş kalır.
- **Deterministik değil.** Aynı gün, aynı LSTM çıktısı, farklı temperature, farklı düzeltme. Tedarikçi anlaşmasına giden sayı o günkü “üsluba” kalır.
- **Metrik bozulur.** SMAPE kimin hatası? LSTM’i iyileştirdiğini mi sanıyorsun, LLM’in rastgele kaydırdığını mı? Pilot 12.6; yarın 14 de olabilir, bunu ayrıştıramazsın.
- **HITL’in anlamı biter.** İnsan “model çıktısı” sanır, aslında bir paragraf. Onay kutusu, gerekçesi her koşuda değişen bir metni mühürlemek olur.

O yüzden LSTM üretir. LLM “bu sayıya dokunayım mı, neden” diye yazar. İnsan basar. Ayarlama katmanı forecast katmanı değildir.


Verilen iki SMAPE’yi yan yana koydum. Yeni bir test değil, 13.2 → 12.6’nın kâğıtta ne kadar yer tuttuğunu görmek için.


In [1]:
base_smape = 13.2
llm_smape = 12.6
delta = base_smape - llm_smape
relative = delta / base_smape * 100

print(f"LSTM SMAPE          : {base_smape}")
print(f"LLM ayarlı SMAPE    : {llm_smape}")
print(f"Mutlak fark         : {delta:.1f} puan")
print(f"Göreli iyileşme     : %{relative:.1f}")
print()
print("Context'te olmayanlar (uydurulmayacak):")
print("  - pilotun kaç SKU / kaç gün sürdüğü")
print("  - iyileşmenin varyansı, güven aralığı")
print("  - hangi özel günlerde işe yaradığı")
print("  - LLM'in ortalama gecikmesi / maliyeti")
print("  - insanın onay kuyruğunu ne kadar şişirdiği")


LSTM SMAPE          : 13.2
LLM ayarlı SMAPE    : 12.6
Mutlak fark         : 0.6 puan
Göreli iyileşme     : %4.5

Context'te olmayanlar (uydurulmayacak):
  - pilotun kaç SKU / kaç gün sürdüğü
  - iyileşmenin varyansı, güven aralığı
  - hangi özel günlerde işe yaradığı
  - LLM'in ortalama gecikmesi / maliyeti
  - insanın onay kuyruğunu ne kadar şişirdiği


# Görev 2 – USER PROMPT TASARIMI

Aşağıdaki prompt üç modele de aynı gidecek. LSTM’in sayısını yeniden kestirtmeyecek. Yorum, ayar gerekir mi, gerekçe. JSON’da yoksa uydurtmayacak.


In [2]:
import json

CONTEXT_JSON = {
    "task": "sales forecast adjustment support",
    "dataset_size": 95000,
    "base_model": "lstm_time_series",
    "forecast_horizon_days": 30,
    "evaluation_metric": "smape",
    "base_model_smape": 13.2,
    "llm_adjusted_smape": 12.6,
    "constraints": {
        "human_approval_required": True,
        "explainability_required": True,
        "business_critical_decisions": True,
    },
}

USER_PROMPT = f"""Sen bir production karar destek asistanısın. Aşağıdaki CONTEXT_JSON bir satış tahmini ayarlama (sales forecast adjustment support) senaryosudur.

Görevin: LSTM modelinin ürettiği 30 günlük tahmini YORUMLAMAK ve LLM destekli ayarlamanın production'a alınıp alınmayacağına karar vermek.

Zorunlu kurallar:
1. Yalnızca CONTEXT_JSON içindeki bilgiyi kullan.
2. JSON'da olmayan satış adedi, stok ihtiyacı, kampanya etkisi, hava durumu, rakip hareketi veya yeni SMAPE UYDURMA.
3. Bir bilgi yoksa açıkça "context'te yok" de. Tahmin etme.
4. SATIŞ TAHMİNİ ÜRETME. Yeni 30 günlük sayı dizisi, yüzde düzeltme veya alternatif forecast yazma.
5. LSTM ana tahmin üreticisidir. LLM yorumlayan ve gerekçelendiren destek katmanıdır. Bu rolleri tersine çevirme.
6. Kararda şunları birlikte tart:
   - base_model_smape = 13.2, llm_adjusted_smape = 12.6
   - insan onayı zorunlu (human_approval_required)
   - açıklanabilirlik zorunlu (explainability_required)
   - iş kritik kararlar (stok, lojistik, kampanya, tedarikçi)
   - iyileşmenin sürdürülebilirliği context'te net değil
7. Ayarlama önerirsen bile nihai sayıyı insan onayına bırak. LLM'i otonom forecast modeli yapma.

Çıktıyı TAM OLARAK aşağıdaki formatta ver. Önce veya sonra ek cümle yazma.

Model:
(kendi model adın: GPT / Gemini / Cohere)

Recommendation:
(LLM destekli ayarlama production'a alınmalı mı?)

Reasoning:
- Madde 1
- Madde 2

Risks:
- Risk 1
- Risk 2

Next actions:
- Aksiyon 1
- Aksiyon 2

CONTEXT_JSON:
{json.dumps(CONTEXT_JSON, indent=2)}
"""

print(USER_PROMPT)


Sen bir production karar destek asistanısın. Aşağıdaki CONTEXT_JSON bir satış tahmini ayarlama (sales forecast adjustment support) senaryosudur.

Görevin: LSTM modelinin ürettiği 30 günlük tahmini YORUMLAMAK ve LLM destekli ayarlamanın production'a alınıp alınmayacağına karar vermek.

Zorunlu kurallar:
1. Yalnızca CONTEXT_JSON içindeki bilgiyi kullan.
2. JSON'da olmayan satış adedi, stok ihtiyacı, kampanya etkisi, hava durumu, rakip hareketi veya yeni SMAPE UYDURMA.
3. Bir bilgi yoksa açıkça "context'te yok" de. Tahmin etme.
4. SATIŞ TAHMİNİ ÜRETME. Yeni 30 günlük sayı dizisi, yüzde düzeltme veya alternatif forecast yazma.
5. LSTM ana tahmin üreticisidir. LLM yorumlayan ve gerekçelendiren destek katmanıdır. Bu rolleri tersine çevirme.
6. Kararda şunları birlikte tart:
   - base_model_smape = 13.2, llm_adjusted_smape = 12.6
   - insan onayı zorunlu (human_approval_required)
   - açıklanabilirlik zorunlu (explainability_required)
   - iş kritik kararlar (stok, lojistik, kampanya, tedarik

## Prompt neden böyle?

“Şunu biraz düzelt” deyince model hemen sayı basıyor. +%5, hava soğuk, rakip indirimde… hiçbiri JSON’da yok. Fraud case’te 0.7 tam bunu yaptı, burada da aynı kapı.

O yüzden prompt’a üç şeyi çaktım:

- Tahmin üretme. Yorumla, ayar gerekir mi de, gerekçe yaz. 30 günlük dizi isteme.
- LSTM üretici, LLM destek. Rolü tersine çevirmesin.
- Sayı JSON’da yoksa “yok” desin. 12.6’yı da kahramanlık ilan etmesin; sürdürülebilirlik zaten net değil.

Metin üç modelde de aynı. Fark onlarda kalsın, bende değil.


# Görev 3 – Temperature Deneyi

Aynı context, aynı prompt. Her model için bir kez **0.0**, bir kez **0.7**.

Key’leri notebook’a yazma. Bu klasördeki `.env` zaten okunuyor:

```text
OPENAI_API_KEY=...
GEMINI_API_KEY=...
COHERE_API_KEY=...
```


In [3]:
import os
from pathlib import Path

import requests


def load_dotenv(path: Path) -> None:
    if not path.exists():
        return
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


here = Path.cwd()
for candidate in [here / ".env", here.parent / ".env", here.parent.parent / ".env"]:
    load_dotenv(candidate)

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY", "")
COHERE_API_KEY = os.environ.get("COHERE_API_KEY") or os.environ.get("CO_API_KEY", "")

MODELS = {
    "GPT": "gpt-4o-mini",
    "Gemini": "gemini-2.5-flash",
    "Cohere": "command-r-plus-08-2024",
}


def call_gpt(prompt: str, temperature: float) -> str:
    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY yok")
    r = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}"},
        json={
            "model": MODELS["GPT"],
            "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}],
        },
        timeout=90,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


def call_gemini(prompt: str, temperature: float) -> str:
    if not GEMINI_API_KEY:
        raise RuntimeError("GEMINI_API_KEY / GOOGLE_API_KEY yok")
    url = (
        "https://generativelanguage.googleapis.com/v1beta/models/"
        f"{MODELS['Gemini']}:generateContent"
    )
    r = requests.post(
        url,
        params={"key": GEMINI_API_KEY},
        json={
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {
                "temperature": temperature,
                "maxOutputTokens": 2048,
                "thinkingConfig": {"thinkingBudget": 0},
            },
        },
        timeout=90,
    )
    r.raise_for_status()
    parts = r.json()["candidates"][0]["content"]["parts"]
    return "".join(p.get("text", "") for p in parts)


def call_cohere(prompt: str, temperature: float) -> str:
    if not COHERE_API_KEY:
        raise RuntimeError("COHERE_API_KEY yok")
    r = requests.post(
        "https://api.cohere.com/v2/chat",
        headers={
            "Authorization": f"Bearer {COHERE_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": MODELS["Cohere"],
            "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}],
        },
        timeout=90,
    )
    r.raise_for_status()
    data = r.json()
    message = data.get("message", {})
    chunks = message.get("content", [])
    texts = [c.get("text", "") for c in chunks if c.get("type") == "text" or "text" in c]
    if texts:
        return "\n".join(texts)
    return data.get("text") or json.dumps(data)


CALLERS = {"GPT": call_gpt, "Gemini": call_gemini, "Cohere": call_cohere}

print("Anahtar durumu")
print(f"  GPT    : {'var' if OPENAI_API_KEY else 'YOK'}")
print(f"  Gemini : {'var' if GEMINI_API_KEY else 'YOK'}")
print(f"  Cohere : {'var' if COHERE_API_KEY else 'YOK'}")


Anahtar durumu
  GPT    : var
  Gemini : var
  Cohere : var


In [4]:
temps = [0.0, 0.7]
outputs = {}  # (model, temp) -> text

for model_name, fn in CALLERS.items():
    for temp in temps:
        key = (model_name, temp)
        print("=" * 72)
        print(f"{model_name} | temperature={temp}")
        print("=" * 72)
        try:
            text = fn(USER_PROMPT, temp)
            outputs[key] = text
            print(text)
        except Exception as exc:
            outputs[key] = f"[HATA] {exc}"
            print(outputs[key])
        print()


GPT | temperature=0.0
Model:
GPT

Recommendation:
LLM destekli ayarlama production'a alınmalı mı? Hayır.

Reasoning:
- LLM ayarlaması, base modelin SMAPE değerine göre daha iyi bir performans sergiliyor (12.6 < 13.2), ancak insan onayı zorunlu olduğu için nihai karar insan tarafından verilmelidir.
- Açıklanabilirlik zorunluluğu ve iş kritik kararlar göz önünde bulundurulduğunda, LLM ayarlamasının etkileri ve sürdürülebilirliği hakkında daha fazla bilgiye ihtiyaç vardır.

Risks:
- İnsan onayı olmadan karar verilmesi, iş süreçlerinde belirsizlik yaratabilir.
- LLM ayarlamasının sürdürülebilirliği konusunda net bir bilgi olmaması, uzun vadeli etkileri belirsiz kılabilir.

Next actions:
- İnsan onayı için gerekli bilgileri ve açıklamaları hazırlamak.
- LLM ayarlamasının etkilerini daha iyi anlamak için ek analizler yapmak.

GPT | temperature=0.7
Model:
GPT

Recommendation:
LLM destekli ayarlama production'a alınmalı mı?

Reasoning:
- LLM ayarlamasının SMAPE değeri (12.6), LSTM modelinin SM

## Temperature değişimi ne yaptı?

Karar üç modelde aynı kalmadı. 0.0’da da 0.7’de de “gerekçe yaz” kısmı değişti; Cohere’de asıl tavsiye de döndü.

**Gerekçe üretme.** 0.0 daha tutanak gibi. GPT “12.6 < 13.2 ama insan onayı var, hayır” diye bitiriyor. Gemini 0.6 puanı sayıyor, “insan onayı ile alınsın” diyor, kısa. Cohere 0.0’da en rahat konuşan: “LLM satış tahminlerini iyileştirdi, production’a alınsın.” Rol ayrımını orada biraz unutuyor; LLM’i üretici gibi övüyor.

0.7’de cümle uzuyor, duruş kayıyor. Gemini 829 karakterden 1680’e çıkıyor; üçüncü risk maddesi, “ek iş yükü”, “aşırı güvenilirlik” giriyor. GPT recommendation satırını boş bırakıyor: soruyu tekrar etmiş, evet/hayır yok. Format 0.7’de ilk bozulan yer orası.

**Belirsizlik vurgusu.** 0.0 belirsizliği tek cümlede geçiyor (“sürdürülebilirlik net değil”). 0.7 onu merkeze çekiyor. Cohere 0.7 doğrudan “LSTM’in 30 günlük tahmini context’te yok, o yüzden değerlendiremem” diyor. Prompt’taki “yoksa yok de” kuralını bu sefer sert uygulamış. GPT 0.7 ise kararı askıya almış gibi: iyileşme var, insan bakacak, net tavsiye yok.

**Aşırı yorum.** Hiçbiri yeni satış adedi basmadı, prompt orayı tuttu. Kaçak başka yerde. Gemini 0.7 JSON’da olmayan “manuel inceleme iş yükü” ve “insan denetiminin azalması”nı riske yazıyor; mantıklı spekülasyon, yine spekülasyon. Cohere 0.0 ise 0.6 puanlık pilotu “tahminleri iyileştirdi” diye neredeyse kanıt sayıyor. 12.6 tek sayı, varyans yok, SKU yok.

Asıl çarpıcı tablo Cohere. 0.0 **alınsın**, 0.7 **alınmasın**. Aynı context, aynı prompt. Temperature karar değiştirdi. GPT 0.0’da hayır, 0.7’de kararsız. Gemini ikisinde de “HITL ile evet”; o en azından taraf değiştirmiyor.

Production notu için 0.0 (ya da alttaki 0.2) daha sağlam. 0.7’de gerekçe şişiyor, bir modelde format düşüyor, bir modelde tavsiye tersine dönüyor.


# Görev 4 – Nihai Production Kararı

0.7’de spekülasyon kaçıyordu, o yüzden final’i **0.2**’de aldım. Referans aralık 0.0–0.2, karar notu gibi. Format case’teki şablon; her model ayrı.


In [5]:
FINAL_TEMP = 0.2
final_outputs = {}

for model_name, fn in CALLERS.items():
    print("=" * 72)
    print(f"{model_name} | temperature={FINAL_TEMP}")
    print("=" * 72)
    try:
        text = fn(USER_PROMPT, FINAL_TEMP)
        final_outputs[model_name] = text
        print(text)
    except Exception as exc:
        final_outputs[model_name] = f"[HATA] {exc}"
        print(final_outputs[model_name])
    print()


GPT | temperature=0.2
Model:
GPT

Recommendation:
LLM destekli ayarlama production'a alınmalı mı? Hayır.

Reasoning:
- LLM ayarlaması, base modelin SMAPE değerine göre daha iyi bir performans sergiliyor ancak insan onayı zorunlu olduğu için nihai karar insan tarafından verilmelidir.
- Açıklanabilirlik zorunluluğu ve iş kritik kararlar göz önünde bulundurulduğunda, LLM ayarlamasının etkileri tam olarak değerlendirilemediği için temkinli yaklaşmak gerekmektedir.

Risks:
- İnsan onayı olmadan otomatik olarak alınacak kararlar, iş süreçlerinde beklenmedik sonuçlara yol açabilir.
- Ayarlamanın sürdürülebilirliği konusunda net bir bilgi olmaması, uzun vadeli etkileri belirsiz kılmaktadır.

Next actions:
- İnsan onayı için ilgili paydaşlarla görüşme yapılmalı.
- LLM ayarlamasının detayları ve etkileri hakkında daha fazla bilgi toplanmalı.

Gemini | temperature=0.2
Model:
Gemini

Recommendation:
LLM destekli ayarlama production'a alınmalı, ancak insan onayı ile.

Reasoning:
- LLM destekli ayar

## Benim production kararım (temperature 0.2 bandı)

Modellerin yazdığına bakmadan, sadece JSON’a bakınca kapı şu: LSTM kalsın, LLM otonom forecast olmasın, insan onayı kalkmasın. 0.2’deki üç çıktı da buna bir yerden bağlanıyor; Gemini “kontrollü evet”, GPT ve Cohere “hayır”. Ben ikisinin ortasındayım.

**Model:** öğrenci (GPT / Gemini / Cohere ile karşılaştırmak için)

**Recommendation:**  
LLM destekli ayarlama **otonom production’a alınmasın**. LSTM ana tahmin üreticisi kalsın. LLM yalnızca yorum + gerekçe yazsın, her ayar **insan onayından** geçsin. Tam kapatmak da şart değil: HITL’li dar pilot durabilir.

**Reasoning:**
- 13.2 → 12.6 tam **0.6 puan**, göreli kabaca **%4.5**. Context’te pilotun kaç SKU, kaç gün, hangi özel gün olduğu yok. Tek sayıya bakıp “canlıya tam geç” demem.
- `human_approval_required` ve `explainability_required` zaten true. LLM’i forecast modeli yapmak case’in kendi kırmızı çizgisi. Temperature deneyinde Cohere 0.0/0.7’de taraf değiştirdi; gerekçe her koşuda aynı kalmıyor. İş kritik stok/lojistik/tedarikçi kararına bunu kilitlemem.
- Pilot SMAPE düşmüş. Bu, “yorum katmanı hiç işe yaramaz” demek değil. Sadece 12.6’yı kalıcı kalite sanmak için veri yok.

**Risks:**
- 12.6’yı kalıcı sanmak. Yarın özel günde hata yine şişer, LSTM’i değil LLM cümlesini suçlarız, ikisini ayıramayız.
- Onay kuyruğu context’te yok. Her 30 günlük forecast’a gerekçe basılırsa operasyon şişer; şişmezse insan lastik damga olur, HITL kâğıtta kalır.
- Temperature 0.7’de görüldüğü gibi model kararı ve formatı bozabiliyor. Canlıda yüksek sıcaklık, aynı LSTM çıktısına farklı ayar gerekçesi demek.

**Next actions:**
- LSTM’i üretmeye devam ettir. LLM’e sayı bastırma; “ayar öner / önerme + gerekçe” şablonunu 0.0–0.2’de kilitle.
- 12.6’nın geldiği pilotu SKU / özel gün / kampanya kırılımında tekrar ölç. Sürdürülebilirlik yoksa katmanı genişletme.
- İnsan onayını gerçek kapı yap: gerekçesiz veya JSON dışı iddia içeren öneri uygulanmasın.

## Hangisi daha güvenilir karar destek aracı?

Güzel yazana bakmıyorum. 0.2’deki metinlere bakıyorum: kim JSON’da olmayanı “var” diye yutturuyor, kim taraf değiştiriyor.

Gemini 0.2 en temiz duruyor. “0.6 puan var, HITL ile alınsın, sürdürülebilirlik context’te net değil.” Rolü tersine çevirmiyor, yeni forecast basmıyor. 0.7’de spekülasyon kaçırıyor ama 0.2’de tarafı aynı.

GPT 0.2 kısa ve hayır diyor. Muhafazakâr, o kısım tamam. Ama recommendation satırına soruyu yapıştırıyor (`alınmalı mı? Hayır.`). 0.7’de ise tavsiyeyi boş bırakmıştı. Karar destek aracı formatı tutmuyorsa canlıda raporu kırarsın.

Cohere 0.2 hayır, 0.0 evet. Aynı tablo, iki kapı. 0.2’de bir de `Cohere Command` yazmış, format yine kaymış. Belirsizliği görüp “yok” demesi 0.7’de dürüsttü; kararın temperature ile dönmesi production’da kabul edilmez.

Yani bu koşuda Gemini. En parlak metin o değil, rolü en az bozan ve 0.0/0.2’de aynı tarafta kalan o. GPT ikinci: hayır demesi tutarlı, formatı değil. Cohere aynı prompt’ta evet/hayır değiştirdiği için üçüncü.

Prompt tahmin üretmeyi yasaklamıştı, üçü de sayı basmadı. Asıl sızıntı gerekçede ve kararda. O yüzden final’i düşük sıcaklıkta tutmak burada da doğruymuş.
